In [ ]:
# Script adapted from: https://github.com/Sr933/rcc/tree/main

import pandas as pd
import os
import numpy as np
import loompy

def make_df(data_path):
    df = pd.read_csv(data_path)
    id = "Gene-ID" if "RCC7" in data_path else "Gene_ID"
    df = df.drop(columns=[id]).set_index('Symbol')

    # Handle duplicate gene names by making them unique
    if df.index.duplicated().any():
        print(f"  Found {df.index.duplicated().sum()} duplicate genes, making unique...")
        df = df.groupby(df.index).sum()  # Option 1: Sum expression values for duplicates
        # Alternative option 2: Make index unique by appending suffix
        # df.index
    return df

main_folder = r"path_to\GSE156632_RAW"

# Initialize lists to store data and labels
all_dataframes = []
all_cell_types = []
all_cell_ids = []

# Loop through files in the main folder
for file in os.listdir(main_folder):
    print(file)
    data_path = os.path.join(main_folder, file)
    
    # Create dataframe from the file
    ar = make_df(data_path)
    all_dataframes.append(ar)  # Keep as genes x cells (don't transpose)
    
    # Extract column names (cell IDs)
    cell_ids = ar.columns.tolist()
    all_cell_ids.extend(cell_ids)
    
    # Read cell type annotations
    file_base_name = os.path.splitext(os.path.basename(file))[0]
    annot_folder_path = r"path_to\China\CellAnnotations"
    second_csv_path = os.path.join(annot_folder_path, f"{file_base_name}.csvcell_annotation.csv")

    cell_type_data = pd.read_csv(second_csv_path)
    
    # Create dictionary mapping cell IDs to cell types
    label_dict = pd.Series(cell_type_data.iloc[:, -1].values, 
                          index=cell_type_data.iloc[:, 0].values).to_dict()
    
    # Get cell types for all cells in this matrix
    cell_types = [label_dict[cell_id] for cell_id in cell_ids]
    all_cell_types.extend(cell_types)
    

# Combine all matrices (concatenate along columns/cells axis)
# combined_df = pd.concat(all_dataframes, axis=1, join='inner')  # 'inner' keeps only common genes
combined_df = pd.concat(all_dataframes, axis=1, join='outer').fillna(0)

# print(f"Using {len(combined_df)} common genes across all samples")
combined_matrix = combined_df.to_numpy()

# Get gene names from the last processed dataframe
gene_names = combined_df.index.tolist()

# Validate shapes
print("Matrix shape (genes x cells):", combined_matrix.shape)
print("Number of cells:", len(all_cell_ids))
print("Number of cell types:", len(all_cell_types))

# Create loom file
output_path = r"path_to\GSE156632_RAW\all_cells_all_genes.loom"

# Row attributes (genes)
row_attrs = {
    "Gene": np.array(gene_names)
}

# Column attributes (cells)
col_attrs = {
    "CellID": np.array(all_cell_ids),
    "CellType": np.array(all_cell_types)
}

# Create the loom file
loompy.create(output_path, combined_matrix, row_attrs, col_attrs)

print(f"Loom file created successfully at: {output_path}")

GSM4735364_RCC1t.csv
  Found 5 duplicate genes, making unique...
GSM4735365_RCC1n.csv
  Found 9 duplicate genes, making unique...
GSM4735366_RCC2t.csv
  Found 9 duplicate genes, making unique...
GSM4735367_RCC2n.csv
  Found 10 duplicate genes, making unique...
GSM4735368_RCC3t.csv
  Found 6 duplicate genes, making unique...
GSM4735369_RCC3n.csv
  Found 6 duplicate genes, making unique...
GSM4735370_RCC4t.csv
  Found 12 duplicate genes, making unique...
GSM4735371_RCC4n.csv
  Found 7 duplicate genes, making unique...
GSM4735372_RCC5t.csv
  Found 6 duplicate genes, making unique...
GSM4735373_RCC5n.csv
  Found 5 duplicate genes, making unique...
GSM4735374_RCC6t.csv
  Found 8 duplicate genes, making unique...
GSM4735375_RCC7t.csv
  Found 10 duplicate genes, making unique...
Matrix shape (genes x cells): (23928, 54988)
Number of cells: 54988
Number of cell types: 54988
Loom file created successfully at: C:\Users\james\scRNA\RawDatasets\China\GSE156632_RAW\all_cells_all_genes.loom
